# 5.1 数据并行 (Data Parallelism)

数据并行是最基础的分布式训练方式，将数据批次分配到多个设备，各设备持有完整模型副本并行计算。是所有分布式训练策略的起点。

本节涵盖：
- DP与DDP原理与通信分析
- Ring-AllReduce通信原语
- FSDP完全分片数据并行
- ZeRO优化三阶段
- DDP最佳实践

## 1. DP与DDP原理

**DP（DataParallel）**：PyTorch最早的并行方式，使用参数服务器模式。GPU 0作为master收集梯度、更新参数、广播给其他GPU。通信瓶颈在GPU 0。

**DDP（DistributedDataParallel）**：使用Ring-AllReduce，每个GPU只与相邻GPU通信，通信量均匀分布。是PyTorch推荐的并行方式。

**核心流程**：
1. 每张GPU持有完整模型副本
2. 数据批次均分到各GPU（micro_batch = global_batch / n_gpus）
3. 各GPU独立前向+反向传播
4. AllReduce同步梯度（DDP使用Ring-AllReduce）
5. 各GPU独立更新参数（参数相同因为梯度相同）

**限制**：模型必须能放入单GPU显存！DDP不减少单卡显存需求。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print('=== DP vs DDP Communication Analysis ===')

model_params_b = 7
bytes_per_param = 2
model_size_gb = model_params_b * bytes_per_param
grad_size_gb = model_size_gb

print(f'Model: {model_params_b}B params, gradient size: {grad_size_gb:.0f} GB')

print(f'\n--- DP (Parameter Server) ---')
for n_gpus in [2, 4, 8]:
    send_per_gpu = grad_size_gb
    recv_per_gpu = grad_size_gb
    gpu0_total = send_per_gpu * (n_gpus - 1) + recv_per_gpu * (n_gpus - 1)
    print(f'  {n_gpus} GPUs: GPU0 sends+receives {gpu0_total:.0f} GB, others {send_per_gpu + recv_per_gpu:.0f} GB each')
    print(f'    GPU0 bottleneck: {gpu0_total/(send_per_gpu+recv_per_gpu):.1f}x load imbalance')

print(f'\n--- DDP (Ring-AllReduce) ---')
for n_gpus in [2, 4, 8, 16]:
    data_per_step = grad_size_gb * 2 * (n_gpus - 1) / n_gpus
    total_per_gpu = data_per_step
    print(f'  {n_gpus} GPUs: each GPU sends+receives {total_per_gpu:.1f} GB (balanced)')

print(f'\n--- Memory per GPU (BF16 training with AdamW) ---')
for model_b in [1.5, 7, 13, 70]:
    params_gb = model_b * 2
    grads_gb = model_b * 2
    optimizer_gb = model_b * 12
    activation_gb = model_b * 2
    total = params_gb + grads_gb + optimizer_gb + activation_gb
    fits_80gb = '✓' if total <= 80 else '✗'
    print(f'  {model_b:>5.1f}B model: params={params_gb:.0f}GB + grads={grads_gb:.0f}GB + opt={optimizer_gb:.0f}GB + act≈{activation_gb:.0f}GB = {total:.0f}GB {fits_80gb}')

print(f'\nKey: DDP balances communication across all GPUs (no bottleneck).')
print(f'But each GPU still needs the FULL model + optimizer state!')
print(f'7B model needs ~112GB per GPU → must use ZeRO/FSDP for A100-80GB.')

In [ ]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def ddp_demo():
    print('=== Minimal DDP Example (run with: torchrun --nproc_per_node=2 this_script.py) ===')
    print('Setup steps:')
    print('  1. dist.init_process_group(backend="nccl")')
    print('  2. local_rank = int(os.environ["LOCAL_RANK"])')
    print('  3. torch.cuda.set_device(local_rank)')
    print('  4. model = DDP(model, device_ids=[local_rank])')
    print('  5. Use DistributedSampler for data loading')
    print('  6. Only rank 0 saves checkpoints / logs')
    print('  7. dist.barrier() for synchronization')
    print('  8. dist.destroy_process_group() at the end')

ddp_demo()
print(f'\nKey: DDP is PyTorch\'s recommended approach for single-node multi-GPU training.')
print(f'For multi-node: use the same code with different MASTER_ADDR/MASTER_PORT.')

## 2. Ring-AllReduce通信原语

**目的**：理解DDP的核心通信算法

**Ring-AllReduce**：将GPU排成逻辑环，数据分块后在环上传递，分两个阶段：

1. **Reduce-Scatter**：每个GPU将梯度分成N块，在环上逐块求和，最终每个GPU持有一块完整的求和结果
2. **All-Gather**：每个GPU将自己持有的求和结果块在环上传递，最终所有GPU都拥有完整的求和梯度

**通信量**：每个GPU发送和接收 2×(N-1)/N × 数据量，与GPU数量近似无关

**带宽利用率**：环上所有GPU同时通信，带宽利用率100%

In [ ]:
import torch

torch.manual_seed(42)

print('=== Ring-AllReduce Simulation ===')

n_gpus = 4
n_chunks = n_gpus
chunk_size = 4

grads = [torch.randn(n_chunks, chunk_size) for _ in range(n_gpus)]
print('Initial gradients (per GPU):')
for i, g in enumerate(grads):
    print(f'  GPU {i}: {g[0, :3].tolist()}...')

shards = [g.clone() for g in grads]

print(f'\n--- Phase 1: Reduce-Scatter ({n_gpus-1} steps) ---')
for step in range(n_gpus - 1):
    new_shards = [s.clone() for s in shards]
    for gpu_id in range(n_gpus):
        sender = (gpu_id - step - 1) % n_gpus
        chunk_idx = (gpu_id - step) % n_gpus
        new_shards[gpu_id][chunk_idx] += shards[sender][chunk_idx]
    shards = new_shards
    print(f'  Step {step+1}: chunks accumulated')

print(f'\nAfter Reduce-Scatter (each GPU has one fully reduced chunk):')
for i, s in enumerate(shards):
    print(f'  GPU {i}: chunk {i} reduced = {s[i, :3].tolist()}...')

expected_sum = sum(grads)
for i in range(n_gpus):
    diff = (shards[i][i] - expected_sum[i]).abs().max().item()
    print(f'  GPU {i} chunk {i} vs expected: max_diff={diff:.6f}')

print(f'\n--- Phase 2: All-Gather ({n_gpus-1} steps) ---')
for step in range(n_gpus - 1):
    new_shards = [s.clone() for s in shards]
    for gpu_id in range(n_gpus):
        sender = (gpu_id - step - 1) % n_gpus
        chunk_idx = (sender - step) % n_gpus
        new_shards[gpu_id][chunk_idx] = shards[sender][chunk_idx]
    shards = new_shards

print(f'After All-Gather (all GPUs have full reduced gradient):')
for i, s in enumerate(shards):
    diff = (s - expected_sum).abs().max().item()
    print(f'  GPU {i}: matches expected sum = {diff < 1e-5}')

print(f'\nKey: Ring-AllReduce achieves O(2×(N-1)/N × data) communication per GPU.')
print(f'All GPUs communicate simultaneously → full bandwidth utilization.')
print(f'NCCL implements this efficiently with GPU Direct RDMA.')

## 3. FSDP与ZeRO优化

**目的**：突破单卡显存瓶颈，训练超大模型

**ZeRO三个阶段**：
- **ZeRO-1**：分片优化器状态（Adam的m和v），节省最大（优化器约占60%显存）
- **ZeRO-2**：+分片梯度，进一步节省
- **ZeRO-3/FSDP**：+分片模型参数，每卡只需1/N的完整状态

**FSDP工作流程**（ZeRO-3）：
1. 每个GPU只存储1/N的参数分片
2. 计算某层时，AllGather聚合该层完整参数
3. 执行前向/反向计算
4. 计算完成后立即释放非本地分片
5. 反向传播时ReduceScatter梯度到各GPU

**通信开销**：ZeRO-3的通信量是DDP的1.5倍（AllGather参数 + ReduceScatter梯度）

In [ ]:
import torch

print('=== ZeRO Optimization: Detailed Memory Analysis ===')

models = [
    ('1.5B (GPT-2 XL)', 1.5),
    ('7B (LLaMA-7B)', 7),
    ('13B (LLaMA-13B)', 13),
    ('70B (LLaMA-70B)', 70),
]

for name, params_b in models:
    print(f'\n--- {name} ---')
    params_gb = params_b * 2
    grads_gb = params_b * 2
    opt_gb = params_b * 12
    total = params_gb + grads_gb + opt_gb
    print(f'Total state: {total:.0f} GB (params={params_gb:.0f}, grads={grads_gb:.0f}, opt={opt_gb:.0f})')

    for n_gpus in [1, 8, 64]:
        if n_gpus == 1:
            z0 = total
            z1 = total
            z2 = total
            z3 = total
        else:
            z0 = total
            z1 = params_gb + grads_gb + opt_gb / n_gpus
            z2 = params_gb + grads_gb / n_gpus + opt_gb / n_gpus
            z3 = params_gb / n_gpus + grads_gb / n_gpus + opt_gb / n_gpus
        fits = lambda m: '✓' if m <= 80 else '✗'
        print(f'  {n_gpus:2d} GPUs: ZeRO-0={z0:6.1f}GB {fits(z0)}, ZeRO-1={z1:6.1f}GB {fits(z1)}, '
              f'ZeRO-2={z2:6.1f}GB {fits(z2)}, ZeRO-3={z3:6.1f}GB {fits(z3)}')

print(f'\n--- Communication Overhead ---')
print(f'ZeRO stage | Extra communication vs DDP')
print(f'ZeRO-0     | 1.0x (baseline DDP)')
print(f'ZeRO-1     | 1.0x (only optimizer sharded, no extra comm)')
print(f'ZeRO-2     | 1.0x (gradients already communicated in DDP)')
print(f'ZeRO-3     | 1.5x (AllGather params + ReduceScatter grads)')

print(f'\nKey: ZeRO-1 gives the biggest win with NO extra communication cost.')
print(f'ZeRO-3/FSDP enables training models >80GB but costs 1.5x communication.')
print(f'Rule of thumb: Use ZeRO-1/2 first, only use ZeRO-3 if model does not fit.')

## 4. DDP最佳实践

**目的**：掌握DDP在生产环境中的关键配置和排错技巧

**关键配置**：
- **bucket_cap_mb**：梯度分桶大小，默认25MB。小桶→更多通信-计算重叠，大桶→更少通信延迟
- **find_unused_parameters**：如果模型有条件执行（如loss mask），必须设为True，否则设为False（False更快）
- **gradient_as_bucket_view**：True可节省显存（梯度直接存在桶中），但梯度值在backward后被覆盖

**常见问题**：
- **死锁**：所有GPU等待彼此→检查是否有GPU提前退出或异常
- **梯度NaN**：某GPU产生NaN→AllReduce后所有GPU梯度变NaN→需要全局检测
- **NCCL超时**：GPU间通信超时→检查网络、IB配置、防火墙
- **Checkpoint不一致**：保存/加载时rank 0写入，其他rank等待同步

**启动方式**：
```bash
torchrun --nproc_per_node=8 train.py
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

class DDPGradientSimulator:
    def __init__(self, n_gpus=4, bucket_cap_mb=25, model_params_mb=1000):
        self.n_gpus = n_gpus
        self.bucket_cap_mb = bucket_cap_mb
        self.model_params_mb = model_params_mb
        self.n_buckets = max(1, model_params_mb // bucket_cap_mb)
        self.bucket_size_mb = model_params_mb / self.n_buckets

    def simulate_backward(self, compute_per_bucket_ms=2.0, comm_per_bucket_ms=1.5, overlap=True):
        if overlap:
            total_ms = self.n_buckets * max(compute_per_bucket_ms, comm_per_bucket_ms) + comm_per_bucket_ms
        else:
            total_ms = self.n_buckets * (compute_per_bucket_ms + comm_per_bucket_ms)
        return total_ms

model_params_mb = 14000

print('=== DDP Best Practices ===')

print(f'\n--- Bucket Size Impact (7B model, {model_params_mb}MB gradients) ---')
print(f'{"Bucket(MB)":>12} {"#Buckets":>10} {"No Overlap(ms)":>16} {"With Overlap(ms)":>18} {"Speedup":>10}')
for bucket_mb in [5, 10, 25, 50, 100, 500]:
    sim = DDPGradientSimulator(n_gpus=8, bucket_cap_mb=bucket_mb, model_params_mb=model_params_mb)
    no_overlap = sim.simulate_backward(overlap=False)
    with_overlap = sim.simulate_backward(overlap=True)
    speedup = no_overlap / with_overlap
    print(f'{bucket_mb:>12} {sim.n_buckets:>10} {no_overlap:>16.1f} {with_overlap:>18.1f} {speedup:>10.2f}x')

print(f'\n--- Gradient NaN Detection ---')
grads = [torch.randn(100) for _ in range(4)]
grads[2][0] = float('nan')

allreduce_grad = sum(grads) / len(grads)
has_nan = torch.isnan(allreduce_grad).any().item()
print(f'GPU 2 has NaN gradient → After AllReduce: all GPUs have NaN = {has_nan}')
print(f'Solution: Check NaN before AllReduce, skip the step if detected.')

print(f'\n--- Checkpoint Consistency ---')
print(f'Best practice for distributed checkpointing:')
print(f'  1. Only rank 0 saves the model state_dict')
print(f'  2. All ranks barrier() before and after saving')
print(f'  3. Use torch.save with _use_new_zipfile_serialization=True')
print(f'  4. For FSDP: use model.state_dict() with full_state_dict=True')

print(f'\n--- torchrun Example ---')
print(f'# Single node, 8 GPUs:')
print(f'torchrun --nproc_per_node=8 train.py')
print(f'\n# Multi-node (2 nodes × 8 GPUs):')
print(f'# Node 0 (master):')
print(f'torchrun --nnodes=2 --nproc_per_node=8 --master_addr=10.0.0.1 --master_port=29500 train.py')
print(f'# Node 1:')
print(f'torchrun --nnodes=2 --nproc_per_node=8 --master_addr=10.0.0.1 --master_port=29500 train.py')

print(f'\nKey: bucket_cap_mb=25 is a good default; tune for your model size.')
print(f'Always check for NaN gradients before AllReduce to prevent training corruption.')
print(f'Use torchrun (not torch.distributed.launch) for production training.')